In [123]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_score, recall_score, f1_score

In [99]:
x_train= joblib.load('../data/x_train.pkl')
x_test= joblib.load('../data/x_test.pkl')
y_train= joblib.load('../data/y_train.pkl')
y_test= joblib.load('../data/y_test.pkl')
columntransformer= joblib.load('../models/preprocessor.pkl')


In [100]:
random_forest_model = Pipeline(steps=[
    ('preprocessor', columntransformer),
    ('classifier', (RandomForestClassifier(
     class_weight="balanced", random_state=42)))
])

In [101]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [6,10,None]}

In [102]:
cv =  StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gridsearch = GridSearchCV(estimator=random_forest_model, param_grid=param_grid, cv=cv, scoring='roc_auc',  n_jobs=-1, refit=True)

In [103]:
print(type(x_train))
print(x_train.columns)

<class 'pandas.DataFrame'>
Index(['order_id', 'product_category', 'price_inr', 'discount_pct',
       'payment_method', 'customer_tenure_days', 'num_previous_orders',
       'num_previous_returns', 'delivery_distance_km', 'delivery_days',
       'is_weekend_order', 'rating_given'],
      dtype='str')


In [104]:
gridsearch.fit(x_train, y_train)
best_cv_score = gridsearch.best_score_
best_params = gridsearch.best_params_

In [107]:
#refit best estimator and eun on x_test to get predictions
best_estimator = gridsearch.best_estimator_
y_pred = best_estimator.predict(x_test)
roc_auc = roc_auc_score(y_test, y_pred)
print(f"Best Cross validated ROC_AUC Score: {best_cv_score}")
print(f"Best Parameter combination is: {best_params}")
print(f"ROC AUC Score on Test Set: {roc_auc}")

Best Cross validated ROC_AUC Score: 0.6186963444710865
Best Parameter combination is: {'classifier__max_depth': 6, 'classifier__n_estimators': 100}
ROC AUC Score on Test Set: 0.5916758538117761


In [115]:
# feature importance
feature_importance = best_estimator.named_steps['classifier'].feature_importances_
feature_names = best_estimator.named_steps['preprocessor'].get_feature_names_out()

print("Feature Importance:")
top_five_features = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False).head(5)

print("Top 5 Features:")
print(top_five_features)


Feature Importance:
Top 5 Features:
                      Feature  Importance
14    cat__payment_method_COD    0.172006
0              num__price_inr    0.138374
4   num__customer_tenure_days    0.094756
3           num__discount_pct    0.093861
7   num__delivery_distance_km    0.091597


Returns are higher in features like payment method, price, tenure days, discount and distance. Cash on Delivery makes it easy for customers to refuse the parcel since they haven’t paid upfront. Price plays a role because expensive items raise expectations, and if those aren’t met, people send them back. New customers, with less experience on the platform, tend to return more as they experiment. Heavy discounts often trigger impulse purchases that lead to regret later. Finally, longer delivery distances increase delays or damage risk, making customers more likely to reject or return the order.

In [117]:


# Run permutation importaance

perm_result = permutation_importance(
    best_estimator,     
    x_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Build a dataframe of raw (pre-one-hot) feature names vs their permutation importance
perm_importance_df = pd.DataFrame({
    "Feature": x_test.columns,
    "Perm_Importance_Mean": perm_result.importances_mean,
    "Perm_Importance_Std": perm_result.importances_std
}).sort_values(by="Perm_Importance_Mean", ascending=False)

print("Permutation Importance (all features):")
print(perm_importance_df)

# Compare side by side:  top-5 impurity features vs their permutation rank
top5_impurity_raw_names = ["payment_method", "price_inr", "customer_tenure_days",
                            "discount_pct", "delivery_distance_km"]

comparison = perm_importance_df[perm_importance_df["Feature"].isin(top5_impurity_raw_names)].copy()
comparison["Impurity_Rank"] = comparison["Feature"].apply(lambda f: top5_impurity_raw_names.index(f) + 1)
comparison["Perm_Rank"] = comparison["Perm_Importance_Mean"].rank(ascending=False).astype(int)

print("\nSide-by-side comparison (impurity rank vs permutation rank):")
print(comparison[["Feature", "Impurity_Rank", "Perm_Importance_Mean", "Perm_Rank"]])

Permutation Importance (all features):
                 Feature  Perm_Importance_Mean  Perm_Importance_Std
4         payment_method              0.093234             0.009141
7   num_previous_returns              0.006854             0.002670
1       product_category              0.005883             0.004883
2              price_inr              0.004675             0.005336
9          delivery_days              0.002926             0.005114
10      is_weekend_order              0.000051             0.000824
0               order_id              0.000000             0.000000
3           discount_pct             -0.000797             0.002480
8   delivery_distance_km             -0.000890             0.002487
6    num_previous_orders             -0.001324             0.002505
11          rating_given             -0.004826             0.001942
5   customer_tenure_days             -0.009745             0.002202

Side-by-side comparison (impurity rank vs permutation rank):
               

Among the original top-5 features (payment_method, price_inr, customer_tenure_days, discount_pct, delivery_distance_km), the one that loses the most importance under permutation is customer_tenure_days, which drops from rank 3 to rank 5 and even shows negative importance. Impurity-based importance can overrate noisy continuous features, which is where permutation importance catches  overrided features reagrless of holding geninue impact.

In [122]:
#precison and recall
results = []

# Overall performance
overall_precision = precision_score(y_test, y_pred)
overall_recall = recall_score(y_test, y_pred)

print("Overall Precision:", overall_precision)
print("Overall Recall:", overall_recall)

# Breakout by product_category
for cat in x_test['product_category'].unique():
    mask = x_test['product_category'] == cat
    prec = precision_score(y_test[mask], y_pred[mask])
    rec = recall_score(y_test[mask], y_pred[mask])
    results.append({'group':'product_category', 'value':cat,
                    'precision':prec, 'recall':rec})

# Breakout by payment_method
for pm in x_test['payment_method'].unique():
    mask = x_test['payment_method'] == pm
    prec = precision_score(y_test[mask], y_pred[mask])
    rec = recall_score(y_test[mask], y_pred[mask])
    results.append({'group':'payment_method', 'value':pm,
                    'precision':prec, 'recall':rec})

# Convert to DataFrame for inspection
df_results = pd.DataFrame(results)
print(df_results)

Overall Precision: 0.31101511879049676
Overall Recall: 0.5274725274725275
              group         value  precision    recall
0  product_category          Home   0.226804  0.647059
1  product_category   Electronics   0.285714  0.346154
2  product_category      Footwear   0.361702  0.607143
3  product_category       Apparel   0.309524  0.520000
4  product_category        Beauty   0.439024  0.580645
5    payment_method           COD   0.320366  0.903226
6    payment_method  Prepaid_Card   0.200000  0.040816
7    payment_method   Prepaid_UPI   0.333333  0.041667
8    payment_method        Wallet   0.000000  0.000000


One subgroup that performs meaningfully worse than the overall average is product_category = "electronics", where recall is substantially lower than the overall recall.  A concrete next step would be to apply a category-specific threshold for electronics predictions, lowering the decision cutoff to boost recall.

In [125]:
#sweep logic from logistic regression notebook
y_proba = best_estimator.predict_proba(x_test)[:, 1]
# Threshold sweep 
thresholds = np.arange(0.1, 0.9, 0.02)
results = []
for threshold in thresholds:
    y_pred_threshold = (y_proba>= threshold).astype(int)
    results.append({
        'threshold': round(threshold, 2),
        'f1_score': f1_score(y_test, y_pred_threshold, pos_label=1),
        'precision': precision_score(y_test, y_pred_threshold, pos_label=1, zero_division=0),
        'recall': recall_score(y_test, y_pred_threshold, pos_label=1)
    })

threshold_results = pd.DataFrame(results)

# Find the F1-maximising threshold
best_threshold_row = threshold_results.loc[threshold_results['f1_score'].idxmax()]
t_star_rf = best_threshold_row['threshold']

print(f"Best Threshold (t*_rf): {t_star_rf}")
print(f"Best F1 Score: {best_threshold_row['f1_score']:.3f}")
print(f"Best Precision: {best_threshold_row['precision']:.3f}")
print(f"Best Recall: {best_threshold_row['recall']:.3f}")

Best Threshold (t*_rf): 0.42
Best F1 Score: 0.398
Best Precision: 0.265
Best Recall: 0.799


In [129]:
joblib.dump(best_estimator, '../models/return_risk_model.pkl')

['../models/return_risk_model.pkl']